In [2]:
import sys
from pathlib import Path

current_dir = Path.cwd().resolve()
# walks upward until it finds a directory containing ppm/
PROJECT_ROOT = next(
    (p for p in (current_dir, *current_dir.parents) if (p / "ppm").is_dir()),
    current_dir,
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from ppm.wandb_utils import load_multiple_experiments

plots_dir = PROJECT_ROOT / "plots/pre-training"
plots_dir.mkdir(parents=True, exist_ok=True)

In [9]:
PROJECTS=[
  "BPI12_001",
  "BPI12_002",
  "BPI12_003",
  "BPI15_001",
  "BPI15_002",
  "BPI15_003",
  "BPI15_005",
  "BPI17_001",
  "BPI17_002",
  "BPI17_003",
  "BPI19_001",
  "BPI19_002",
  "BPI19_003",
  "BPI20_001",
  "BPI20_002",
  "baseline-nep"
]

runs_raw, history_raw = load_multiple_experiments(PROJECTS, force_update=False)

runs_raw["best_test_final_next_activity_f1"] = (
    runs_raw["best_test_final_next_activity_f1"]
    .combine_first(runs_raw["best_test_final_next_activity_f1_macro"])
)
#runs_raw = runs_raw.drop(columns=["best_test_final_next_activity_f1_macro"])



Database already exists: /app/visualization/paper/metrics/BPI12_001.db
Use force_update=True to re-fetch from wandb
Database already exists: /app/visualization/paper/metrics/BPI12_002.db
Use force_update=True to re-fetch from wandb
Database already exists: /app/visualization/paper/metrics/BPI12_003.db
Use force_update=True to re-fetch from wandb
Database already exists: /app/visualization/paper/metrics/BPI15_001.db
Use force_update=True to re-fetch from wandb
Database already exists: /app/visualization/paper/metrics/BPI15_002.db
Use force_update=True to re-fetch from wandb
Database already exists: /app/visualization/paper/metrics/BPI15_003.db
Use force_update=True to re-fetch from wandb
Database already exists: /app/visualization/paper/metrics/BPI15_005.db
Use force_update=True to re-fetch from wandb
Database already exists: /app/visualization/paper/metrics/BPI17_001.db
Use force_update=True to re-fetch from wandb
Database already exists: /app/visualization/paper/metrics/BPI17_002.db
U

In [10]:
import pandas as pd

pd.set_option("display.max_columns", None)   # show all columns
pd.set_option("display.width", None)         # don't wrap/truncate by width
pd.set_option("display.max_colwidth", None)  # optional: full cell content

runs_raw.columns

Index(['id', 'name', 'r', 'lr', 'log', 'device', 'epochs', 'compile',
       'n_heads', 'backbone', 'n_layers', 'patience', 'rnn_type', 'strategy',
       'val_size', 'grad_clip', 'lifecycle', 'min_delta', 'precision',
       'val_split', 'batch_size', 'lora_alpha', 'fine_tuning', 'hidden_size',
       'num_workers', 'total_params', 'weight_decay', 'weight_tying',
       'freeze_layers', 'embedding_size', 'trainable_params',
       'continuous_targets', 'categorical_targets', 'continuous_features',
       'categorical_features', 'memory_safety_margin',
       'time_positional_encoding', 'duration_sec', 'best_test_final_loss',
       'best_test_final_next_activity_acc', 'best_test_final_next_activity_f1',
       'best_test_final_next_activity_loss', 'dataset/filtered_dataset_cases',
       'dataset/filtered_dataset_end', 'dataset/filtered_dataset_events',
       'dataset/filtered_dataset_max_duration_days',
       'dataset/filtered_dataset_start', 'dataset/orig_dataset_cases',
       'd

In [11]:
# Detemrmine best val_loss (these were not recorded during run time, only test set metrics were recorded, but we can determine the best val_loss post-hoc)

VAL_LOSS_COL = "val_loss"

# ---- compute best (min) validation loss per run ----
tmp = history_raw.dropna(subset=[VAL_LOSS_COL]).copy()
idx = tmp.groupby(["project", "run_id"])[VAL_LOSS_COL].idxmin()
best_val = tmp.loc[idx, ["project", "run_id", "step", VAL_LOSS_COL]].rename(
    columns={VAL_LOSS_COL: "best_val_loss", "step": "best_val_loss_step"}
)

## ---- join back onto runs ----
#runs = runs_raw.merge(
#    best_val,
#    left_on=["project", "id"],
#    right_on=["project", "run_id"],
#    how="left",
#)#

In [12]:
best_val.head()

,project,run_id,best_val_loss_step,best_val_loss
459,BPI12_001,14lul5nl,7.0,0.514299
523,BPI12_001,31s3a140,8.0,0.674029
34,BPI12_001,4qksvez2,34.0,0.764264
413,BPI12_001,5bq18ofo,11.0,0.457864
553,BPI12_001,5gnt3dfs,18.0,0.807766


In [40]:
runs.head()

,id,name,r,lr,log,device,epochs,compile,n_heads,backbone,n_layers,patience,rnn_type,strategy,val_size,grad_clip,lifecycle,min_delta,precision,val_split,batch_size,lora_alpha,fine_tuning,hidden_size,num_workers,total_params,weight_decay,weight_tying,freeze_layers,embedding_size,trainable_params,continuous_targets,categorical_targets,continuous_features,categorical_features,memory_safety_margin,time_positional_encoding,duration_sec,best_test_final_loss,best_test_final_next_activity_acc,best_test_final_next_activity_f1,best_test_final_next_activity_loss,dataset/filtered_dataset_cases,dataset/filtered_dataset_end,dataset/filtered_dataset_events,dataset/filtered_dataset_max_duration_days,dataset/filtered_dataset_start,dataset/orig_dataset_cases,dataset/orig_dataset_end,dataset/orig_dataset_events,dataset/orig_dataset_max_duration_days,dataset/orig_dataset_start,dataset/test_cases,dataset/test_end,dataset/test_events,dataset/test_max_duration_days,dataset/test_start,dataset/train_cases,dataset/train_end,dataset/train_events,dataset/train_max_duration_days,dataset/train_start,dataset/train_test_intersect_cases,dataset/train_test_intersect_end,dataset/train_test_intersect_max_duration_days,dataset/train_test_intersect_start,dataset/train_test_separation_time,dataset/train_val_intersect_cases,dataset/train_val_intersect_end,dataset/train_val_intersect_max_duration_days,dataset/train_val_intersect_start,dataset/train_val_separation_time,dataset/val_cases,dataset/val_end,dataset/val_events,dataset/val_max_duration_days,dataset/val_start,oom_batch_count,test_loss,test_next_activity_acc,test_next_activity_f1,test_next_activity_loss,train_loss,train_next_activity_acc,train_next_activity_f1,train_next_activity_loss,val_loss,val_next_activity_acc,val_next_activity_f1,val_next_activity_loss,created_at,tags,group,job_type,project,seed,prediction_table,transition_table,best_test_final_next_activity_f1_macro
0,4qksvez2,charmed-hill-1,2.0,0.005,BPI12,cuda,250.0,True,1.0,gpt2-mini,1.0,10.0,lstm,sum,0.1,5.0,True,0.005,bf16,prefix,64.0,4.0,freeze,128.0,8.0,38659623.0,0.1,True,"1,-1",512.0,6360103.0,None,[activity],"[accumulated_time, amount]","[activity, resource]",None,None,163.626982,0.686878,0.797832,0.664330,0.686878,9487.0,2012-02-28,173498.0,32.0,2011-09-30,13087.0,2012-03-14,262200.0,137.0,2011-09-30,1965.0,2012-02-28,35332.0,31.0,2011-12-13,7264.0,2012-01-02,124404.0,32.0,2011-09-30,633.0,2012-02-13,32.0,2011-12-12,2012-01-12,450.0,2012-01-12,32.0,2011-12-02,2012-01-02,1341.0,2012-01-12,13762.0,30.0,2011-12-08,0.0,0.966156,0.759085,0.631487,0.966156,0.413070,0.841532,0.703786,0.413070,1.045099,0.729400,0.577566,1.045099,2026-02-24 16:03:37+00:00,[],None,None,BPI12_001,NaN,NaN,NaN,NaN
1,855xxk2x,feasible-field-2,2.0,0.005,BPI12,cuda,250.0,True,1.0,gpt2-mini,1.0,10.0,lstm,sum,0.1,5.0,True,0.005,bf16,prefix,64.0,4.0,freeze,128.0,8.0,38659623.0,0.1,True,"1,-1",512.0,6360103.0,None,[activity],"[accumulated_time, amount]","[activity, resource]",None,None,96.097881,0.606325,0.811078,0.665228,0.606325,9487.0,2012-02-28,173498.0,32.0,2011-09-30,13087.0,2012-03-14,262200.0,137.0,2011-09-30,1965.0,2012-02-28,35332.0,31.0,2011-12-13,7264.0,2012-01-02,124404.0,32.0,2011-09-30,633.0,2012-02-13,32.0,2011-12-12,2012-01-12,450.0,2012-01-12,32.0,2011-12-02,2012-01-02,1341.0,2012-01-12,13762.0,30.0,2011-12-08,0.0,0.779412,0.805162,0.666741,0.779412,0.334799,0.861484,0.737450,0.334799,0.840231,0.782517,0.606130,0.840231,2026-02-24 16:06:34+00:00,[],None,None,BPI12_001,NaN,NaN,NaN,NaN
2,a64vdfh8,fresh-aardvark-3,2.0,0.005,BPI12,cuda,250.0,True,1.0,gpt2-mini,1.0,10.0,lstm,sum,0.1,5.0,True,0.005,bf16,prefix,64.0,4.0,freeze,128.0,8.0,38659623.0,0.1,True,"1,-1",512.0,6360103.0,None,[activity],"[accumulated_time, amount]","[activity, resource]",None,None,132.840446,0.497712,0.830805,0.697460,0.497712,9487.0,2012-02-28,173498.0,32.0,2011-09-30,13087.0,2012-03-14,262200.0,137.0,2011-09-30,1965.0,2012-02-28,35332.0,31.0,2011-12-13,7264.0,2012-01-02,124404.0,32.0,2011-09-

In [13]:
# prepocessing
runs = runs_raw.copy()

fine_tuning_col = "fine_tuning" if "fine_tuning" in runs.columns else "fine-tuning"
freeze_layers_col = "freeze_layers" if "freeze_layers" in runs.columns else "freeze-layers"
mask_no_finetuning = runs[fine_tuning_col].isna() | (runs[fine_tuning_col].astype(str) == "None")
runs.loc[mask_no_finetuning, freeze_layers_col] = "all"

for col in ["hidden_size", "n_layers", "n_heads"]:
    if col in runs.columns:
        runs[col] = pd.to_numeric(runs[col], errors="coerce")

def _fmt_int(v):
    return str(int(v)) if pd.notna(v) else "nan"

def _arch_label(df: pd.DataFrame) -> pd.Series:
    hs = df["hidden_size"].map(_fmt_int).astype("string")
    nl = df["n_layers"].map(_fmt_int).astype("string")
    nh = df["n_heads"].map(_fmt_int).astype("string")
    return hs.str.cat([nl, nh], sep="_")

# Keep internal backbone distinct per source for separate aggregation,
# but display as <hidden>_<layers>_<heads> in plots/tables.
scratch_mask = runs["backbone"].astype(str).str.contains("student", na=False)
runs.loc[scratch_mask, "backbone"] = "nano_" + _arch_label(runs.loc[scratch_mask])

backbone_rename_map = {
    "BPI20PrepaidTravelCosts": "BPI20_PTC",
    "BPI20TravelPermitData": "BPI20_TPD",
    "BPI20RequestForPayment": "BPI20_RfP",
}
runs["log"] = runs["log"].replace(backbone_rename_map)

# ---- join back onto runs ----
runs = runs_raw.merge(
    best_val,
    left_on=["project", "id"],
    right_on=["project", "run_id"],
    how="left",
)#


In [15]:
# Ensure backbone is string
runs["backbone"] = runs["backbone"].astype(str)

FIELDS = [
    "log",
    "best_val_loss",
    "best_test_final_next_activity_acc",
    "best_test_final_next_activity_f1",
    "project",
    #"backbone",
    "best_run_id",
    "scope",
    "backbone",  # remove if you don’t want to display it
]

def pick_best_by_log(df: pd.DataFrame) -> pd.DataFrame:
    df = df.dropna(subset=["log", "best_val_loss"]).copy()
    if df.empty:
        return df
    idx = df.groupby("log")["best_val_loss"].idxmin()
    out = df.loc[idx].copy()
    out["best_run_id"] = out["id"]
    return out

# Masks
mask_gpt = runs["backbone"].str.startswith(("gpt2", "qwen3"), na=False)
mask_non_gpt = ~mask_gpt

# 1) All backbones
best_all = pick_best_by_log(runs)
best_all["scope"] = "all_backbones"

# 2) GPT2 / Qwen3 only
best_gpt = pick_best_by_log(runs[mask_gpt])
best_gpt["scope"] = "gpt2_or_qwen3"

# 3) Non-GPT2 / Non-Qwen3
best_non_gpt = pick_best_by_log(runs[mask_non_gpt])
best_non_gpt["scope"] = "non_gpt2_or_qwen3"

# Combine (up to 3 rows per log)
summary = (
    pd.concat([best_gpt, best_non_gpt, best_all], ignore_index=True)
    .sort_values(["log", "scope"])
)

summary = summary[[c for c in FIELDS if c in summary.columns]]

def highlight_gpt(row):
    if row["scope"] == "gpt2_or_qwen3":
        return ["color: green;"] * len(row)
    return [""] * len(row)

styled_summary = summary.style.apply(highlight_gpt, axis=1)

styled_summary

,log,best_val_loss,best_test_final_next_activity_acc,best_test_final_next_activity_f1,project,best_run_id,scope,backbone
14,BPI12,0.365196,0.861570,0.727538,BPI12_002,dhh757qc,all_backbones,student_model
0,BPI12,0.424407,0.852089,0.713256,BPI12_001,9ih1t9kg,gpt2_or_qwen3,gpt2-small
7,BPI12,0.365196,0.861570,0.727538,BPI12_002,dhh757qc,non_gpt2_or_qwen3,student_model
15,BPI15,3.665808,0.189675,0.070170,BPI15_005,oro5tnuw,all_backbones,gpt2-mini
1,BPI15,3.665808,0.189675,0.070170,BPI15_005,oro5tnuw,gpt2_or_qwen3,gpt2-mini
8,BPI15,3.733275,0.167141,0.063727,BPI15_002,s5gmfsom,non_gpt2_or_qwen3,student_model
16,BPI17,0.291696,0.841505,0.682942,BPI17_002,9j4c43g6,all_backbones,student_model
2,BPI17,0.363943,0.829804,0.677057,BPI17_003,f3a4184u,gpt2_or_qwen3,qwen3-1.7b
9,BPI17,0.291696,0.841505,0.682942,BPI17_002,9j4c43g6,non_gpt2_or_qwen3,student_model
17,BPI19,0.561452,0.857794,0.289070,BPI19_002,88rgwbdb,all_backbones,student_model


In [50]:
# Sanity check: for each run, take the row where val_loss is minimal and also extract test metrics at that step

VAL_LOSS_COL = "val_loss"
TEST_ACC_COL = "test_next_activity_acc"
TEST_F1_COL  = "test_next_activity_f1"

tmp = history_raw.dropna(subset=[VAL_LOSS_COL]).copy()

# row (per run) where val_loss is minimal
idx = tmp.groupby(["project", "run_id"])[VAL_LOSS_COL].idxmin()

best_at_min_val = (
    tmp.loc[idx, ["project", "run_id", "step", VAL_LOSS_COL, TEST_ACC_COL, TEST_F1_COL]]
    .rename(
        columns={
            "step": "best_val_loss_step",
            VAL_LOSS_COL: "best_val_loss",
            TEST_ACC_COL: "test_acc_at_best_val",
            TEST_F1_COL: "test_f1_at_best_val",
        }
    )
    .reset_index(drop=True)
)

# Optional: join to runs_raw for run name/backbone/log/etc.
sanity = runs_raw.merge(
    best_at_min_val,
    left_on=["project", "id"],
    right_on=["project", "run_id"],
    how="left",
)

#best_at_min_val, sanity
sanity.head()

,id,name,r,lr,log,device,epochs,compile,n_heads,backbone,n_layers,patience,rnn_type,strategy,val_size,grad_clip,lifecycle,min_delta,precision,val_split,batch_size,lora_alpha,fine_tuning,hidden_size,num_workers,total_params,weight_decay,weight_tying,freeze_layers,embedding_size,trainable_params,continuous_targets,categorical_targets,continuous_features,categorical_features,memory_safety_margin,time_positional_encoding,duration_sec,best_test_final_loss,best_test_final_next_activity_acc,best_test_final_next_activity_f1,best_test_final_next_activity_loss,dataset/filtered_dataset_cases,dataset/filtered_dataset_end,dataset/filtered_dataset_events,dataset/filtered_dataset_max_duration_days,dataset/filtered_dataset_start,dataset/orig_dataset_cases,dataset/orig_dataset_end,dataset/orig_dataset_events,dataset/orig_dataset_max_duration_days,dataset/orig_dataset_start,dataset/test_cases,dataset/test_end,dataset/test_events,dataset/test_max_duration_days,dataset/test_start,dataset/train_cases,dataset/train_end,dataset/train_events,dataset/train_max_duration_days,dataset/train_start,dataset/train_test_intersect_cases,dataset/train_test_intersect_end,dataset/train_test_intersect_max_duration_days,dataset/train_test_intersect_start,dataset/train_test_separation_time,dataset/train_val_intersect_cases,dataset/train_val_intersect_end,dataset/train_val_intersect_max_duration_days,dataset/train_val_intersect_start,dataset/train_val_separation_time,dataset/val_cases,dataset/val_end,dataset/val_events,dataset/val_max_duration_days,dataset/val_start,oom_batch_count,test_loss,test_next_activity_acc,test_next_activity_f1,test_next_activity_loss,train_loss,train_next_activity_acc,train_next_activity_f1,train_next_activity_loss,val_loss,val_next_activity_acc,val_next_activity_f1,val_next_activity_loss,created_at,tags,group,job_type,project,seed,prediction_table,transition_table,best_test_final_next_activity_f1_macro,run_id,best_val_loss_step,best_val_loss,test_acc_at_best_val,test_f1_at_best_val
0,4qksvez2,charmed-hill-1,2.0,0.005,BPI12,cuda,250.0,True,1.0,gpt2-mini,1.0,10.0,lstm,sum,0.1,5.0,True,0.005,bf16,prefix,64.0,4.0,freeze,128.0,8.0,38659623.0,0.1,True,"1,-1",512.0,6360103.0,None,[activity],"[accumulated_time, amount]","[activity, resource]",None,None,163.626982,0.686878,0.797832,0.664330,0.686878,9487.0,2012-02-28,173498.0,32.0,2011-09-30,13087.0,2012-03-14,262200.0,137.0,2011-09-30,1965.0,2012-02-28,35332.0,31.0,2011-12-13,7264.0,2012-01-02,124404.0,32.0,2011-09-30,633.0,2012-02-13,32.0,2011-12-12,2012-01-12,450.0,2012-01-12,32.0,2011-12-02,2012-01-02,1341.0,2012-01-12,13762.0,30.0,2011-12-08,0.0,0.966156,0.759085,0.631487,0.966156,0.413070,0.841532,0.703786,0.413070,1.045099,0.729400,0.577566,1.045099,2026-02-24 16:03:37+00:00,[],None,None,BPI12_001,NaN,NaN,NaN,NaN,4qksvez2,34.0,0.764264,0.797832,0.664330
1,855xxk2x,feasible-field-2,2.0,0.005,BPI12,cuda,250.0,True,1.0,gpt2-mini,1.0,10.0,lstm,sum,0.1,5.0,True,0.005,bf16,prefix,64.0,4.0,freeze,128.0,8.0,38659623.0,0.1,True,"1,-1",512.0,6360103.0,None,[activity],"[accumulated_time, amount]","[activity, resource]",None,None,96.097881,0.606325,0.811078,0.665228,0.606325,9487.0,2012-02-28,173498.0,32.0,2011-09-30,13087.0,2012-03-14,262200.0,137.0,2011-09-30,1965.0,2012-02-28,35332.0,31.0,2011-12-13,7264.0,2012-01-02,124404.0,32.0,2011-09-30,633.0,2012-02-13,32.0,2011-12-12,2012-01-12,450.0,2012-01-12,32.0,2011-12-02,2012-01-02,1341.0,2012-01-12,13762.0,30.0,2011-12-08,0.0,0.779412,0.805162,0.666741,0.779412,0.334799,0.861484,0.737450,0.334799,0.840231,0.782517,0.606130,0.840231,2026-02-24 16:06:34+00:00,[],None,None,BPI12_001,NaN,NaN,NaN,NaN,855xxk2x,18.0,0.643689,0.811078,0.665228
2,a64vdfh8,fresh-aardvark-3,2.0,0.005,BPI12,cuda,250.0,True,1.0,gpt2-mini,1.0,10.0,lstm,sum,0.1,5.0,True,0.005,bf16,prefix,64.0,4.0,freeze,128.0,8.0,38659623.0,0.1,True,"1,-1",512.0,6360103.0,None,[activity],"[accumulated_time, amount]","[activity, resource]",None,None,132.840446,0.497712,0.830805,0.697460,0.497712,9487.0,

In [51]:
# Check correspondence between summary-level "best_test_final_*"
# and test metrics at the step of minimal val_loss

import numpy as np

ACC_SUM_COL = "best_test_final_next_activity_acc"
F1_SUM_COL  = "best_test_final_next_activity_f1"

ACC_STEP_COL = "test_acc_at_best_val"
F1_STEP_COL  = "test_f1_at_best_val"

df_check = sanity.copy()

# Only consider rows where both sides are present
acc_mask = df_check[[ACC_SUM_COL, ACC_STEP_COL]].notna().all(axis=1)
f1_mask  = df_check[[F1_SUM_COL,  F1_STEP_COL]].notna().all(axis=1)

acc_match = np.isclose(
    df_check.loc[acc_mask, ACC_SUM_COL],
    df_check.loc[acc_mask, ACC_STEP_COL],
    atol=1e-8,
)

f1_match = np.isclose(
    df_check.loc[f1_mask, F1_SUM_COL],
    df_check.loc[f1_mask, F1_STEP_COL],
    atol=1e-8,
)

print(f"ACC matches: {acc_match.sum()} / {acc_mask.sum()}")
print(f"F1  matches: {f1_match.sum()} / {f1_mask.sum()}")

# Optional: show mismatches for inspection
acc_mismatch_df = df_check.loc[acc_mask].loc[~acc_match, 
    ["project", "id", ACC_SUM_COL, ACC_STEP_COL]
]

f1_mismatch_df = df_check.loc[f1_mask].loc[~f1_match, 
    ["project", "id", F1_SUM_COL, F1_STEP_COL]
]

print("\nACC mismatches:")
print(acc_mismatch_df.head())

print("\nF1 mismatches:")
print(f1_mismatch_df.head())

print("mismatch probably due to use of delta in early stopping")

ACC matches: 277 / 485
F1  matches: 276 / 485

ACC mismatches:
      project        id  best_test_final_next_activity_acc  \
2   BPI12_001  a64vdfh8                           0.830805   
6   BPI12_001  9ih1t9kg                           0.852089   
8   BPI12_001  sstrql85                           0.853759   
9   BPI12_001  liyy6u9g                           0.841051   
12  BPI12_001  5gwp3bl0                           0.828767   

    test_acc_at_best_val  
2               0.833579  
6               0.851579  
8               0.852825  
9               0.844956  
12              0.838560  

F1 mismatches:
      project        id  best_test_final_next_activity_f1  test_f1_at_best_val
2   BPI12_001  a64vdfh8                          0.697460             0.697998
6   BPI12_001  9ih1t9kg                          0.713256             0.720962
8   BPI12_001  sstrql85                          0.719029             0.718249
9   BPI12_001  liyy6u9g                          0.694275             